## 100% Full Data PMT Observables and Event Rates for the ANNIE NCQE Analysis

There are two sets of MC files we can analyze: GENIE + Empirical model for O16, and the GENIE base model. Please ensure the relative paths are correct in your workflow.

The MC ntuples can be found here: `/pnfs/annie/persistent/users/doran/GENIE-016-empirical-model/MC_toolchain`
and
`/pnfs/annie/persistent/users/doran/GENIE-Base/MC_toolchain`

In [ ]:
print('Loading Packages...\n')
from lib.imports import *
from lib.constants import *
from lib.reco_functions import (
    reco_energy,
    bunch_cutting,
    pairwise_relative_direction,
    is_inside_rotated_parabola,
)

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #
# Analyze Base GENIE files ('Base') or Empirical Model ('Empirical')

which_MC, POT = select_MC()
print('\n', POT, 'POT')

# REQUIRES USER INPUT (specify path to MC files)
MC_directory = '../MC_GENIE-O16/'   # or 'MC_GENIE-Base/'
plots_dir = 'plots/'                # directory to hold plots
os.system('mkdir -p ' + plots_dir)  # auto-generate plots directory if not present

print('\nWill use ' + MC_directory + ' for the MC samples, and ' + str(plots_dir) +' for the plots output directory')

data_directory = '../../FY22_23/'
csv_file = 'lib/POT_trigger_FY22_23_summary.csv'   # contains trigger counts and POT for on-beam data
offbeam_directory = '../../off_beam/'
csv_file_ext = 'lib/offbeam_summary.csv'           # contains trigger counts for off-beam data

# simulated skyshine files
skyshine_file_names = [
    '../../skyshine_100k_no_ext_clusters.root',
    '../../skyshine_100k_no_ext_clusters_2.root',
    '../../skyshine_100k_no_ext_clusters_3.root'
]

EXPECTED_SKYSHINE = 14.1            # number of skyshine events expcted in our signal region, determined from the sideband analysis

print('\ndone')

In [ ]:
# Mini-cell w/ pre-loading to determine the total number of true NCQE signal events in the FV

total_weighted = 0.0
total_raw      = 0          # unweighted count, useful sanity check

# These files were not passed through STV, but do contain weights
# You can optionally use the files that have ran through STV (that have the event categories and selections applied)
# But I used the non-STV files for transparency

file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')
    nc  = S['trueNC'].array(library='np')
    qel = S['trueQEL'].array(library='np')
    nu  = S['trueFSLPdg'].array(library='np')
    tz  = S['trueTargetZ'].array(library='np')
    wt  = S['weight_TunedCentralValue_UBGenie'].array(library='np')
    
    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)
    
    # ── single combined selection mask ────────────────────────────────────────
    mask = (
        fv_flags          &
        (nc  == 1)        &
        (qel == 1)        &
        (tz  == 8)        &
        ((nu == 14) | (nu == 12))
    )

    wt_cv = np.concatenate(wt).astype(float)

    total_weighted += wt_cv[mask].sum()
    total_raw      += mask.sum()
    

print(f'\nRaw signal events  : {total_raw}')
print(f'Weighted signal sum: {total_weighted:.4f}')      # CV weights shouldn't matter for NC
print('\ndone')

In [ ]:
# Load MC samples

file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_recoVtx           = []
MC_recoVty           = []
MC_recoVtz           = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_QEL               = []
MC_MEC               = []
MC_PDG               = []
MC_Z                 = []
MC_CV_weight         = []

# jagged accumulators
MC_hitX  = []
MC_hitY  = []
MC_hitZ  = []
MC_hitT  = []
MC_hitPE = []
MC_MRD_hitT  = []
MC_FMV_hitT  = []

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    MC_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    MC_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))

    MC_hitX.append(T['hitX'].array())
    MC_hitY.append(T['hitY'].array())
    MC_hitZ.append(T['hitZ'].array())
    MC_hitT.append(T['hitT'].array())
    MC_hitPE.append(T['hitPE'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_FV.append(fv_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_QEL.append(S['trueQEL'].array(library='np')[CEN])
    MC_MEC.append(S['trueMEC'].array(library='np')[CEN])
    MC_PDG.append(S['trueFSLPdg'].array(library='np')[CEN])
    MC_Z.append(S['trueTargetZ'].array(library='np')[CEN])
    
    MC_CV_weight.append(S['weight_TunedCentralValue_UBGenie'].array(library='np')[CEN])

    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    MC_MRD_hitT.append(mrd_t[CEN])
    MC_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_recoVtx         = np.concatenate(MC_recoVtx)
MC_recoVty         = np.concatenate(MC_recoVty)
MC_recoVtz         = np.concatenate(MC_recoVtz)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_FV              = np.concatenate(MC_FV)
MC_NC              = np.concatenate(MC_NC)
MC_QEL             = np.concatenate(MC_QEL)
MC_MEC             = np.concatenate(MC_MEC)
MC_PDG             = np.concatenate(MC_PDG)
MC_Z               = np.concatenate(MC_Z)
MC_CV_weight       = np.concatenate(MC_CV_weight)

MC_hitX     = ak.concatenate(MC_hitX)
MC_hitY     = ak.concatenate(MC_hitY)
MC_hitZ     = ak.concatenate(MC_hitZ)
MC_hitT     = ak.concatenate(MC_hitT)
MC_hitPE    = ak.concatenate(MC_hitPE)
MC_MRD_hitT = ak.concatenate(MC_MRD_hitT)
MC_FMV_hitT = ak.concatenate(MC_FMV_hitT)

print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

In [ ]:
# on-beam data (can be found here: /pnfs/annie/persistent/users/doran/datasets/NCQE_BEAMCLUSTER_DATA/EXTRACTED_FULL_BEAM_DATA/)

def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number

file_names = [file for file in os.listdir(data_directory) if os.path.isfile(os.path.join(data_directory, file))]
print('There are: ', len(file_names), ' files\n')

# Extract all run numbers from the current files
run_numbers = [extract_run_number(f) for f in file_names]

# POT file
full_pot_df = pd.read_csv(csv_file)

# Filter the dataframe to only include runs that are in the <STRING>/ directory
pot_df = full_pot_df[full_pot_df['RUN'].isin(run_numbers)]

# extract total POT and triggers for only the matched runs
total_POT_TOR860 = sum(pot_df["TOR860_POTe12"])/(1e8)   # convert to e20 POT
total_triggers = sum(pot_df["TOTAL_TRIGGERS"])

cluster_time = []
cluster_charge = []
cluster_Hits = []
cluster_Qb = []
TankMRDCoinc = []
NoVeto = []
only_prompt_cluster = []
MRD_activity = []
Veto_activity = []
Vtx_reco = []
Vty_reco = []
Vtz_reco = []
hitX = []
hitY = []
hitZ = []
hitT = []
hitPE = []
runs = []

counter = 0
print('(', (counter), '/', len(file_names), ')')

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(data_directory, file_name)) as file:
        
        Event = file["data"]
        
        runs.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time.append(Event["cluster_time_BRF"].array(library="np"))
        cluster_charge.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity.append(Event["MRD_activity"].array(library="np"))
        Veto_activity.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco.append(Event["recoVtx"].array(library="np"))
        Vty_reco.append(Event["recoVty"].array(library="np"))
        Vtz_reco.append(Event["recoVtz"].array(library="np"))
        
        hitX.append(Event["hitX"].array())
        hitY.append(Event["hitY"].array())
        hitZ.append(Event["hitZ"].array())
        hitT.append(Event["hitT"].array())
        hitPE.append(Event["hitPE"].array())

        #print(len(cluster_time[-1]), "clusters")
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs = np.concatenate(runs)
cluster_time = np.concatenate(cluster_time)
cluster_charge = np.concatenate(cluster_charge)
cluster_Qb = np.concatenate(cluster_Qb)
cluster_Hits = np.concatenate(cluster_Hits)
TankMRDCoinc = np.concatenate(TankMRDCoinc)
NoVeto = np.concatenate(NoVeto)
only_prompt_cluster = np.concatenate(only_prompt_cluster)
MRD_activity = np.concatenate(MRD_activity)
Veto_activity = np.concatenate(Veto_activity)
Vtx_reco = np.concatenate(Vtx_reco)
Vty_reco = np.concatenate(Vty_reco)
Vtz_reco = np.concatenate(Vtz_reco)

# Jagged arrays
hitX = ak.concatenate(hitX)
hitY = ak.concatenate(hitY)
hitZ = ak.concatenate(hitZ)
hitT = ak.concatenate(hitT)
hitPE = ak.concatenate(hitPE)

print(total_triggers, 'triggers')
print(total_POT_TOR860, 'e20 POT\n')

In [ ]:
# off beam data (can be found here: /pnfs/annie/persistent/users/doran/datasets/NCQE_BEAMCLUSTER_DATA/EXTRACTED_FULL_OFFBEAM_DATA/)
file_names_ext = [file for file in os.listdir(offbeam_directory) if os.path.isfile(os.path.join(offbeam_directory, file))]
print('There are: ', len(file_names_ext), ' files\n')

# POT file
pot_df_ext = pd.read_csv(csv_file_ext)

# extract total POT and triggers
total_ext_triggers = sum(pot_df_ext["TOTAL_TRIGGERS"])

cluster_time_ext = []
cluster_charge_ext = []
cluster_Hits_ext = []
cluster_Qb_ext = []
TankMRDCoinc_ext = []
NoVeto_ext = []
only_prompt_cluster_ext = []
MRD_activity_ext = []
Veto_activity_ext = []
Vtx_reco_ext = []
Vty_reco_ext = []
Vtz_reco_ext = []
hitX_ext = []
hitY_ext = []
hitZ_ext = []
hitT_ext = []
hitPE_ext = []
runs_ext = []

counter = 0
print('(', (counter), '/', len(file_names_ext), ')')

for file_name in file_names_ext:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names_ext), ')')
    
    with uproot.open(os.path.join(offbeam_directory, file_name)) as file:
        
        Event = file["data"]
        
        runs_ext.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time_ext.append(Event["cluster_time"].array(library="np"))
        cluster_charge_ext.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb_ext.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits_ext.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc_ext.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto_ext.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster_ext.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity_ext.append(Event["MRD_activity"].array(library="np"))
        Veto_activity_ext.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco_ext.append(Event["recoVtx"].array(library="np"))
        Vty_reco_ext.append(Event["recoVty"].array(library="np"))
        Vtz_reco_ext.append(Event["recoVtz"].array(library="np"))
        
        hitX_ext.append(Event["hitX"].array())
        hitY_ext.append(Event["hitY"].array())
        hitZ_ext.append(Event["hitZ"].array())
        hitT_ext.append(Event["hitT"].array())
        hitPE_ext.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs_ext = np.concatenate(runs_ext)
cluster_time_ext = np.concatenate(cluster_time_ext)
cluster_charge_ext = np.concatenate(cluster_charge_ext)
cluster_Qb_ext = np.concatenate(cluster_Qb_ext)
cluster_Hits_ext = np.concatenate(cluster_Hits_ext)
TankMRDCoinc_ext = np.concatenate(TankMRDCoinc_ext)
NoVeto_ext = np.concatenate(NoVeto_ext)
only_prompt_cluster_ext = np.concatenate(only_prompt_cluster_ext)
MRD_activity_ext = np.concatenate(MRD_activity_ext)
Veto_activity_ext = np.concatenate(Veto_activity_ext)
Vtx_reco_ext = np.concatenate(Vtx_reco_ext)
Vty_reco_ext = np.concatenate(Vty_reco_ext)
Vtz_reco_ext = np.concatenate(Vtz_reco_ext)

# Jagged arrays
hitX_ext = ak.concatenate(hitX_ext)
hitY_ext = ak.concatenate(hitY_ext)
hitZ_ext = ak.concatenate(hitZ_ext)
hitT_ext = ak.concatenate(hitT_ext)
hitPE_ext = ak.concatenate(hitPE_ext)

print(total_ext_triggers, 'triggers')

In [ ]:
# Apply selection cuts to the MC

MC_reco_E = [[], [], [], [], [], []]
MC_CV_x = [[], [], [], [], [], []]
MC_CV_y = [[], [], [], [], [], []]
MC_CV_z = [[], [], [], [], [], []]
MC_bunch_times = [[], [], [], [], [], []]
MC_vertex_x = [[], [], [], [], [], []]
MC_vertex_y = [[], [], [], [], [], []]
MC_vertex_z = [[], [], [], [], [], []]
MC_hits = [[], [], [], [], [], []]

MC_weights_by_cat = [[], [], [], [], [], []]

# arrays to store events that are "outside" the 1-sigma bunch cuts (for one specific miniboone-styled plot we show)
MC_outside_bunch_by_cat = [[], [], [], [], [], []]
MC_outside_weights_by_cat = [[], [], [], [], [], []]

NCQE_signal_events = 0    # total number of true signal events


for i in trange(len(MC_cluster_time), desc = 'applying NCQE selection cuts to MC...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if MC_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if MC_bunchTimes[i] > time_to_prompt_end:
        continue
    # minimum 10 hits
    if MC_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    # No MRD/FMV correlated activity
    if (
        MC_TankMRDCoinc[i] == 1
        or MC_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in MC_MRD_hitT[i])
        or any(0 <= t <= 300 for t in MC_FMV_hitT[i])
    ):
        continue
        
    # Cluster time within 1.6 us beam spill
    if MC_bunchTimes[i] > bunch_time_end:
        continue
    if MC_bunchTimes[i] < bunch_time_start:
        continue
        
    if MC_bunchTimes[i] > MC_third_spill_cut:
        continue
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(MC_cluster_time)):        # look at the subsequent clusters in this event
        if MC_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if MC_bunchTimes[k] < time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if MC_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut
    if MC_recoVtx[i] < -50:
        continue
    r2 = MC_recoVtx[i]**2 + MC_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(MC_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(MC_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(MC_hitX[i], MC_hitY[i], MC_hitZ[i], MC_hitPE[i], MC_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue

    # assign w = 1.0 if w is inf
    weight = MC_CV_weight[i][0]
    if np.isinf(weight):
        weight = 1.0
    
    if bunch_cutting(MC_bunchTimes[i], centers) == True:
        
        # event categorization
        if MC_External[i] == True:
            indy = 5
        else:
            if MC_FV[i] == True:
                if MC_NC[i] == 1:       # NC
                    if MC_QEL[i] == 1:
                        if MC_Z[i] == 8:
                            if MC_PDG[i] == 14 or MC_PDG[i] == 12:      # nu (mu + e)
                                # nuNCQE
                                indy = 0
                            elif MC_PDG[i] == -14 or MC_PDG[i] == -12:
                                # nubarNCQE
                                indy = 3
                        else:
                            # NC elastic on H (NCother)
                            indy = 1
                    else:
                        # NC, not QEL (NCother)
                        indy = 1
                else:
                    # CC
                    indy = 2
            else:
                # out-of-FV
                indy = 4

        MC_reco_E[indy].append(recoE)
        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
        MC_bunch_times[indy].append(MC_bunchTimes[i])
        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
        MC_hits[indy].append(MC_cluster_Hits[i])
        MC_weights_by_cat[indy].append(weight)
                
               
    # outside bunches
    else:
        
        # event categorization
        if MC_External[i] == True:
            indy = 5
        else:
            if MC_FV[i] == True:
                if MC_NC[i] == 1:       # NC
                    if MC_QEL[i] == 1:
                        if MC_Z[i] == 8:
                            if MC_PDG[i] == 14 or MC_PDG[i] == 12:      # nu (mu + e)
                                # nuNCQE
                                indy = 0
                            elif MC_PDG[i] == -14 or MC_PDG[i] == -12:
                                # nubarNCQE
                                indy = 3
                        else:
                            # NC elastic on H (NCother)
                            indy = 1
                    else:
                        # NC, not QEL (NCother)
                        indy = 1
                else:
                    # CC
                    indy = 2
            else:
                # out-of-FV
                indy = 4
        
        MC_outside_weights_by_cat[indy].append(weight)
        MC_outside_bunch_by_cat[indy].append(MC_bunchTimes[i])
            

print('done')

In [ ]:
# on-beam data

reco_E = []
CV_x = []
CV_y = []
CV_z = []
bunch_times = []
vertex_x = []
vertex_y = []
vertex_z = []
hits = []
reco_runs = []

# outside bunches
outside_bunch_times = []

for i in trange(len(cluster_time), desc = 'applying NCQE selection cuts to on-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity (extracted ntuples have a condition to check the same 300ns veto + MRD hit window as MC)
    if TankMRDCoinc[i] == 1 or NoVeto[i] == 0 or MRD_activity[i] != 0 or Veto_activity[i] != 0:
        continue
        
    # CCB cut
    if cluster_Qb[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco[i] < -50:
        continue
    r2 = Vtx_reco[i]**2 + Vtz_reco[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX[i], hitY[i], hitZ[i], hitPE[i], hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    # bunch cutting
    if cluster_time[i] < bunch_time_start_data:
        continue
    if cluster_time[i] > bunch_time_end_data:
        continue
            
    if cluster_time[i] > data_third_spill_cut:
        continue

    if bunch_cutting(cluster_time[i], centers_data) == False:
        outside_bunch_times.append(cluster_time[i])
        continue
        
    # event selection
    reco_E.append(recoE)
    CV_x.append(cw[0]); CV_y.append(cw[1]); CV_z.append(cw[2])
    bunch_times.append(cluster_time[i])
    vertex_x.append(Vtx_reco[i]); vertex_y.append(Vty_reco[i]); vertex_z.append(Vtz_reco[i])
    hits.append(cluster_Hits[i])
    reco_runs.append(runs[i])
          

print('done')

In [ ]:
# ext data

ext_reco_E = []
ext_CV_x = []
ext_CV_y = []
ext_CV_z = []
ext_bunch_times = []
ext_vertex_x = []
ext_vertex_y = []
ext_vertex_z = []
ext_hits = []
ext_reco_runs = []

ext_outside_bunch_times = []

for i in trange(len(cluster_time_ext), desc = 'applying NCQE selection cuts to off-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster_ext[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc_ext[i] == 1 or NoVeto_ext[i] == 0 or MRD_activity_ext[i] != 0 or Veto_activity_ext[i] != 0:
        continue
        
    # CCB cut
    if cluster_Qb_ext[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco_ext[i] < -50:
        continue
    r2 = Vtx_reco_ext[i]**2 + Vtz_reco_ext[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco_ext[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge_ext[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX_ext[i], hitY_ext[i], hitZ_ext[i], hitPE_ext[i], hitT_ext[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    # bunch cutting
    if cluster_time_ext[i] < bunch_time_start_data:
        continue
    if cluster_time_ext[i] > bunch_time_end_data:
        continue
        
    if cluster_time_ext[i] > data_third_spill_cut:
        continue

    if bunch_cutting(cluster_time_ext[i], centers_data) == False:
        ext_outside_bunch_times.append(cluster_time_ext[i])
        continue
        
    # event selection
    ext_reco_E.append(recoE)
    ext_CV_x.append(cw[0]); ext_CV_y.append(cw[1]); ext_CV_z.append(cw[2])
    ext_bunch_times.append(cluster_time_ext[i])
    ext_vertex_x.append(Vtx_reco_ext[i]); ext_vertex_y.append(Vty_reco_ext[i]); ext_vertex_z.append(Vtz_reco_ext[i])
    ext_hits.append(cluster_Hits_ext[i])
    ext_reco_runs.append(runs_ext[i])
          

print('done')

In [ ]:
# add skyshine samples to populate observable plots

# scalar accumulators
skyshine_cluster_number    = []
skyshine_cluster_time      = []
skyshine_cluster_charge    = []
skyshine_cluster_Qb        = []
skyshine_cluster_Hits      = []
skyshine_TankMRDCoinc      = []
skyshine_NoVeto            = []
skyshine_recoVtx           = []
skyshine_recoVty           = []
skyshine_recoVtz           = []
skyshine_event_number      = []

# jagged accumulators
skyshine_hitX  = []
skyshine_hitY  = []
skyshine_hitZ  = []
skyshine_hitT  = []
skyshine_hitPE = []
skyshine_hitID = []
skyshine_MRD_hitT  = []
skyshine_FMV_hitT  = []

for file_name in skyshine_file_names:
    path = os.path.join(file_name)
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    skyshine_cluster_number.append(T['clusterNumber'].array(library='np'))
    skyshine_cluster_charge.append(T['clusterPE'].array(library='np'))
    skyshine_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    skyshine_cluster_Hits.append(T['clusterHits'].array(library='np'))
    skyshine_cluster_time.append(T['clusterTime'].array(library='np'))
    skyshine_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    skyshine_NoVeto.append(T['NoVeto'].array(library='np'))
    skyshine_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    skyshine_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    skyshine_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
    skyshine_event_number.append(CEN)

    skyshine_hitX.append(T['hitX'].array())
    skyshine_hitY.append(T['hitY'].array())
    skyshine_hitZ.append(T['hitZ'].array())
    skyshine_hitT.append(T['hitT'].array())
    skyshine_hitPE.append(T['hitPE'].array())
    skyshine_hitID.append(T['hitDetID'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']

    # map trigger-level quantities onto clusters via CEN
    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    skyshine_MRD_hitT.append(mrd_t[CEN])
    skyshine_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

skyshine_cluster_number  = np.concatenate(skyshine_cluster_number)
skyshine_cluster_time    = np.concatenate(skyshine_cluster_time)
skyshine_cluster_charge  = np.concatenate(skyshine_cluster_charge)
skyshine_cluster_Qb      = np.concatenate(skyshine_cluster_Qb)
skyshine_cluster_Hits    = np.concatenate(skyshine_cluster_Hits)
skyshine_TankMRDCoinc    = np.concatenate(skyshine_TankMRDCoinc)
skyshine_NoVeto          = np.concatenate(skyshine_NoVeto)
skyshine_recoVtx         = np.concatenate(skyshine_recoVtx)
skyshine_recoVty         = np.concatenate(skyshine_recoVty)
skyshine_recoVtz         = np.concatenate(skyshine_recoVtz)
skyshine_event_number    = np.concatenate(skyshine_event_number)

skyshine_hitX     = ak.concatenate(skyshine_hitX)
skyshine_hitY     = ak.concatenate(skyshine_hitY)
skyshine_hitZ     = ak.concatenate(skyshine_hitZ)
skyshine_hitT     = ak.concatenate(skyshine_hitT)
skyshine_hitPE    = ak.concatenate(skyshine_hitPE)
skyshine_hitID    = ak.concatenate(skyshine_hitID)
skyshine_MRD_hitT = ak.concatenate(skyshine_MRD_hitT)
skyshine_FMV_hitT = ak.concatenate(skyshine_FMV_hitT)

print(f'Total clusters: {len(skyshine_cluster_number)}')
print('\ndone')

In [ ]:
# apply selection cuts to skyshine

skyshine_reco_E = []
skyshine_CV_x = []
skyshine_CV_y = []
skyshine_CV_z = []
skyshine_bunch_times = []
skyshine_vertex_x = []
skyshine_vertex_y = []
skyshine_vertex_z = []
skyshine_hits = []

skyshine_delta_t_bunch = []


# do not apply standard timing cuts since this is kind of an ad-hoc skyshine simulation
for i in trange(len(skyshine_cluster_time), desc = 'applying NCQE selection cuts to simulated skyshine...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if skyshine_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if skyshine_cluster_time[i] > time_to_prompt_end:
        continue
    # minimum 10 hits
    if skyshine_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    if (
        skyshine_TankMRDCoinc[i] == 1
        or skyshine_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in skyshine_MRD_hitT[i])
        or any(0 <= t <= 300 for t in skyshine_FMV_hitT[i])
    ):
        continue
        
    # Add earlier time cut
    if skyshine_cluster_time[i] > MC_third_spill_cut:
        continue
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(skyshine_cluster_time)):        # look at the subsequent clusters in this event
        if skyshine_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if skyshine_cluster_time[k] < time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if skyshine_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut
    if skyshine_recoVtx[i] < -50:
        continue
    r2 = skyshine_recoVtx[i]**2 + skyshine_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(skyshine_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(skyshine_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(skyshine_hitX[i], skyshine_hitY[i], skyshine_hitZ[i], skyshine_hitPE[i], skyshine_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
        
    skyshine_reco_E.append(recoE)
    skyshine_CV_x.append(cw[0]); skyshine_CV_y.append(cw[1]); skyshine_CV_z.append(cw[2])
    skyshine_bunch_times.append(skyshine_cluster_time[i])
    skyshine_vertex_x.append(skyshine_recoVtx[i]); 
    skyshine_vertex_y.append(skyshine_recoVty[i]);
    skyshine_vertex_z.append(skyshine_recoVtz[i])
    skyshine_hits.append(skyshine_cluster_Hits[i])
    
    # impart random position WRT bunch (since its constant in time)
    skyshine_delta_t_bunch.append(random.random()*bunch_sigma*2 - bunch_sigma)


radius_skyshine = np.sqrt(np.array(skyshine_vertex_x)**2 + np.array(skyshine_vertex_z)**2)
    
print(len(skyshine_bunch_times))
print('done')

In [ ]:
# --- 1. Scaling Constants ---
POT_MC = POT/(1e20)

POT_DATA = total_POT_TOR860
TRIG_DATA = total_triggers
TRIG_EXT = total_ext_triggers

# 2. Chain all bunch times to a common t0
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def get_normalized_times(times, runs, weights=None, mc=False):

    norm_times = []
    norm_weights = []

    for i, (t, r) in enumerate(zip(times, runs)):

        if mc:
            tnorm = t - bunch_time_start
        else:
            tnorm = t - bunch_time_start_data

        norm_times.append(tnorm)

        if weights is not None:
            norm_weights.append(weights[i])

    if weights is not None:
        return np.array(norm_times), np.array(norm_weights)

    return np.array(norm_times)


# 1. Process MC (Categories 0-5)
bunches_80_mc = []
bunches_80_weights = []

for i in range(6):

    t, w = get_normalized_times(
        MC_bunch_times[i],
        [0]*len(MC_bunch_times[i]),
        weights=MC_weights_by_cat[i],
        mc=True
    )

    bunches_80_mc.append(t)
    bunches_80_weights.append(w)

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)


# 3. Calculation of radial distance of the reco vertex
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
# For MC (6 categories)
radius_mc = [np.sqrt(np.array(MC_vertex_x[i])**2 + np.array(MC_vertex_z[i])**2) for i in range(6)]
# For On-Beam Data
radius_data = np.sqrt(np.array(vertex_x)**2 + np.array(vertex_z)**2)
# For Off-Beam (EXT) Data
radius_ext = np.sqrt(np.array(ext_vertex_x)**2 + np.array(ext_vertex_z)**2)


# 4. Relative time from nearest bunch center
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def distance_to_bunch_center(cluster_time, centers):
    avg_diff = bunch_offset_mapping
    # Ensure 'centers' is defined in your environment
    shifted_centers = centers - avg_diff 
    diffs = shifted_centers - cluster_time
    idx = np.argmin(np.abs(diffs))
    return diffs[idx]

# --- MC Calculation (Static) ---
# Assuming 'centers' is your MC bunch center array
delta_t_mc = [
    np.array([distance_to_bunch_center(t, centers) for t in cat]) 
    for cat in MC_bunch_times
]

# --- On-Beam Data ---
delta_t_data_list = []
for t in bunch_times:
    delta_t_data_list.append(distance_to_bunch_center(t, centers_data))

delta_t_data = np.array(delta_t_data_list)

# --- Off-Beam Data ---
delta_t_ext_list = []
for t in ext_bunch_times:
    delta_t_ext_list.append(distance_to_bunch_center(t, centers_data))

delta_t_ext = np.array(delta_t_ext_list)


# Relative time from nearest bunch center, for events OUTSIDE the 1-sigma cut
# (lets us further split "outside" into 1-2 sigma and >2 sigma sub-regions)

delta_t_mc_outside = [
    np.array([distance_to_bunch_center(t, centers) for t in cat])
    for cat in MC_outside_bunch_by_cat
]

delta_t_data_outside = np.array(
    [distance_to_bunch_center(t, centers_data) for t in outside_bunch_times]
)

delta_t_ext_outside = np.array(
    [distance_to_bunch_center(t, centers_data) for t in ext_outside_bunch_times]
)
    
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *

print('done')

In [ ]:
# plotting function for PMT observables with skyshine

def plot_with_skyshine(bins, mc_data, on_beam_data, off_beam_data, skyshine_data, 
                       pot_mc, pot_data, trig_ext, trig_data, expected_skyshine_yield,
                       axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                       plot_name, legend_columns=3, SAVE_FIG=True,mc_weights_by_cat=None,verbose=False):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    
    ### NEW: Calculate the weight required to force the skyshine MC to exactly match your expected yield
    if len(skyshine_data) > 0:
        f_sky = expected_skyshine_yield / len(skyshine_data)
    else:
        f_sky = 0.0
    
    # 2. Prepare Prediction Stack (MC + Skyshine + Off-Beam)
    ### NEW: We inject Skyshine right after External (MC) [which is index 0]
    prediction_data = [mc_data[0], skyshine_data] + mc_data[1:] + [off_beam_data]
    
   # --- MC weights including GENIE tuning weights ---
    mc_weights = []

    for i, arr in enumerate(mc_data):

        if mc_weights_by_cat is None:
            genie_w = np.ones(len(arr))
        else:
            genie_w = np.array(mc_weights_by_cat[i])

        mc_weights.append(genie_w * f_pot)

    # Assemble full weight list
    weights = [mc_weights[0],
               np.ones(len(skyshine_data)) * f_sky] + \
              mc_weights[1:] + \
              [np.ones(len(off_beam_data)) * f_ext]
    
    # 3. Aesthetics
    ### NEW: Added Skyshine to labels and colors and increased outlines to 7
    labels_base = ['External (MC)', 'Skyshine', 'Out-of-FV', 'CC', 'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors = ['papayawhip', 'darkmagenta', 'lime', 'darkred', 'cyan', 'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 7 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ALPHA_SCALING
            if verbose:
                print(f"Applied {ALPHA_SCALING} scale to index {i} ({label})")
        ### NEW: Just a helpful printout to confirm the math worked
        if 'Skyshine' in label and verbose:
            print(f"Scaled {len(skyshine_data)} raw skyshine events to exactly {expected_skyshine_yield:.1f} events")

    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0, alpha = 0.65)

    # B. The Outline
    #ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
    #        histtype='step', color=color_outline, linewidth=1)
    x_step = np.repeat(bins, 2)[1:-1]
    y_band_hi = np.repeat(pred_counts + pred_err, 2)
    y_band_lo = np.repeat(np.maximum(pred_counts - pred_err, 0), 2)
    ax.fill_between(x_step, y_band_lo, y_band_hi,
                    facecolor='none', edgecolor='dimgray',
                    linewidth=0, hatch='//', zorder=3,
                    label='MC stats')
    
    # C. Total prediction outline — CHANGED: single bold outline of the full stack
    #    (replaces the per-component stacked step outlines)
    #    ax.stairs requires matplotlib >= 3.4
    ax.stairs(pred_counts, bins, color='black', linewidth=1.8, zorder=4)
    
    
    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data (stats only)', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    ratio = data_counts[mask] / pred_counts[mask]
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2]) 
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    #ax.text(0.17, 1.02, 'Preliminary', transform=ax.transAxes)
    #ax.text(0.70, 1.02, f"{pot_data*100:.2f} " + r"$\times 10^{18}$ POT", transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes)

    plt.tight_layout()
    if SAVE_FIG:
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()


print('Plotting function loaded')

In [ ]:
# Produce PMT observable distributions 
# (just for one flavor of MC, skip to end to export MC info for a different script that will plot Base vs Empirical)

SAVE_FIGURE = True

# GENIE CV tuning weights
mc_weights_stack = [
    MC_weights_by_cat[5],
    MC_weights_by_cat[4],
    MC_weights_by_cat[2],
    MC_weights_by_cat[1],
    MC_weights_by_cat[3],
    MC_weights_by_cat[0]
]

common_args = {
    'pot_mc': POT_MC,
    'pot_data': POT_DATA,
    'trig_ext': TRIG_EXT,
    'trig_data': TRIG_DATA,
    'legend_columns': 3,
    'mc_weights_by_cat': mc_weights_stack,
    'verbose': False,
}

# ADJUST Y LIMITS AND BINNING BY NEED

# reconstructed energy
mc_stack = [MC_reco_E[5], MC_reco_E[4], MC_reco_E[2], MC_reco_E[1], MC_reco_E[3], MC_reco_E[0]]
plot_with_skyshine(
    bins=np.arange(5, 13, 1),
    mc_data=mc_stack,
    on_beam_data=reco_E,
    off_beam_data=ext_reco_E,
    skyshine_data=skyshine_reco_E, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, 
    axis_label='Reconstructed energy [MeV]',
    y_axis_label='Events / 1 MeV bin',
    x_limit=[5, 12],
    y_limit=[0,200],
    xaxis_minor=0.5,
    plot_name=plots_dir + 'full_data_reco_energy.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)


mc_stack_hits = [MC_hits[5], MC_hits[4], MC_hits[2], MC_hits[1], MC_hits[3], MC_hits[0]]
plot_with_skyshine(
    bins=np.arange(10, 50, 4),
    mc_data=mc_stack_hits,
    on_beam_data=hits,
    off_beam_data=ext_hits,
    skyshine_data=skyshine_hits, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='PMT hits',
    y_axis_label='Events / 4 hit bin',
    x_limit=[10, 50],
    y_limit=[0,250],
    xaxis_minor=1,
    plot_name=plots_dir + 'full_data_PMT_hits.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)

mc_stack_r = [radius_mc[5], radius_mc[4], radius_mc[2], radius_mc[1], radius_mc[3], radius_mc[0]]
plot_with_skyshine(
    bins=np.arange(0, 0.8, 0.1),
    mc_data=mc_stack_r,
    on_beam_data=radius_data,
    off_beam_data=radius_ext,
    skyshine_data=radius_skyshine, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed distance to center [m]',
    y_axis_label='Events / 10 cm bin',
    x_limit=[0, 0.7],
    y_limit=[0,300],
    xaxis_minor=0.1,
    plot_name=plots_dir + 'full_data_radius.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)

mc_stack_y = [MC_vertex_y[5], MC_vertex_y[4], MC_vertex_y[2], MC_vertex_y[1], MC_vertex_y[3], MC_vertex_y[0]]
plot_with_skyshine(
    bins=np.arange(-1, 1.25, 0.25),
    mc_data=mc_stack_y,
    on_beam_data=vertex_y,
    off_beam_data=ext_vertex_y,
    skyshine_data=skyshine_vertex_y, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed vertex (Y) [m]',
    y_axis_label='Events / 50 cm bin',
    x_limit=[-1, 1],
    y_limit=[0,175],
    xaxis_minor=0.2,
    plot_name=plots_dir + 'full_data_Y.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)

vars_cv = [('x', MC_CV_x, CV_x, ext_CV_x, skyshine_CV_x),
           ('y', MC_CV_y, CV_y, ext_CV_y, skyshine_CV_y),
           ('z', MC_CV_z, CV_z, ext_CV_z, skyshine_CV_z)]
for name, mc_list, data_list, ext_list, sky_list in vars_cv:
    mc_stack = [mc_list[5], mc_list[4], mc_list[2], mc_list[1], mc_list[3], mc_list[0]]
    plot_with_skyshine(
        bins=np.arange(-1, 1.25, 0.25) if name != 'z' else np.arange(-1, 0.6, 0.2),
        mc_data=mc_stack,
        on_beam_data=data_list,
        off_beam_data=ext_list,
        skyshine_data=sky_list, # <-- Pass the new skyshine simulated energies here
        expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
        axis_label=r'$C_V$ ' + f'({name})',
        y_axis_label='Events / bin',
        x_limit=[-1, 1] if name != 'z' else [-1, 0.4],
        y_limit=[0,200] if name != 'y' else [0, 225] ,
        xaxis_minor=0.1,
        plot_name=f'{plots_dir}full_data_Cv_{name}.png',
        SAVE_FIG = SAVE_FIGURE,
        **common_args
    )
    
mc_stack = [delta_t_mc[5],
            delta_t_mc[4],
            delta_t_mc[2],
            delta_t_mc[1],
            delta_t_mc[3],
            delta_t_mc[0]]

plot_with_skyshine(
    bins=np.array([0,1,2,3,bunch_sigma]),
    mc_data=mc_stack,
    on_beam_data=delta_t_data,
    off_beam_data=delta_t_ext,
    skyshine_data=skyshine_delta_t_bunch, 
    expected_skyshine_yield=EXPECTED_SKYSHINE, 
    axis_label=r'$\Delta t_{\mathrm{nearest\ bunch}}$ [ns]',
    y_axis_label='Events / bin',
    x_limit=[0,bunch_sigma],
    y_limit=[0,175],
    xaxis_minor=0.25,
    plot_name=plots_dir + 'full_data_delta_t.png',
    SAVE_FIG = SAVE_FIGURE,
    **common_args
)

print('done')

In [ ]:
# Additional PMT observable: time since start of the spill

# 1. Data-driven parameters from the skyshine fit from the sideband
t0 = 353.6
t_max = 18.936 * 25 + bunch_sigma # Max time of your plot

# 2. Get the number of skyshine events in your MC sample
# (Replace 'skyshine_cluster_charge' with whatever your main skyshine array is called)
N_skyshine_mc = len(skyshine_cluster_charge)

# 3. Generate the synthetic times via Inverse Transform Sampling
u = np.random.uniform(0, 1, N_skyshine_mc)
skyshine_times = t0 + (t_max - t0) * np.sqrt(u)

# Note: If your plot needs to include the possibility of times extending 
# slightly past t_max due to bin width, just set t_max to the absolute upper edge of your bins.

# plotting
SAVE_FIGURE = True
bin_width = 18.936 * 6  # Buckets N 16 bunches
bins = np.arange(18.936, 18.936 * 25 + bin_width, bin_width)       # skip the first bunche since its a baby bunch


mc_stack = [bunches_80_mc[5], bunches_80_mc[4], bunches_80_mc[2], 
            bunches_80_mc[1], bunches_80_mc[3], bunches_80_mc[0]]

mc_weights_stack = [
    bunches_80_weights[5],
    bunches_80_weights[4],
    bunches_80_weights[2],
    bunches_80_weights[1],
    bunches_80_weights[3],
    bunches_80_weights[0]
]

common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3,
        'mc_weights_by_cat': mc_weights_stack,
        'verbose': False,
    }

plot_with_skyshine(
    bins=bins,
    mc_data=mc_stack,
    on_beam_data=bunches_80_data,
    off_beam_data=bunches_80_ext,
    skyshine_data=skyshine_times, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Time since first bunch [ns]',
    y_axis_label='Events / 6 BNB bunches',
    x_limit=[18.936, 18.936 * 25],
    y_limit=[0,300],
    xaxis_minor=50,
    plot_name=plots_dir + 'full_data_bunches_6.png',
    SAVE_FIG=SAVE_FIGURE,
    **common_args
)

print('done')

In [ ]:
# Last PMT Observable: re-create the miniboone-style bunch plot

mc_weights_stack = [
    MC_weights_by_cat[5], MC_weights_by_cat[4], MC_weights_by_cat[2],
    MC_weights_by_cat[1], MC_weights_by_cat[3], MC_weights_by_cat[0]
]

mc_stack = [delta_t_mc[5], delta_t_mc[4], delta_t_mc[2],
            delta_t_mc[1], delta_t_mc[3], delta_t_mc[0]]

# same 5,4,2,1,3,0 category order as everywhere else, applied to the outside arrays
cat_order = [5, 4, 2, 1, 3, 0]
delta_t_mc_outside_ord = [delta_t_mc_outside[i] for i in cat_order]
mc_weights_outside_ord = [np.array(MC_outside_weights_by_cat[i]) for i in cat_order]

def split_mid_far(delta_t_arrs, weight_arrs, sigma):
    """1-2 sigma ('mid') vs >2 sigma ('far'), per category."""
    mid_vals, mid_w, far_vals, far_w = [], [], [], []
    for dt, w in zip(delta_t_arrs, weight_arrs):
        dt, w = np.asarray(dt), np.asarray(w)
        m_mid = (np.abs(dt) >= sigma) & (np.abs(dt) < 2 * sigma)
        m_far = np.abs(dt) >= 2 * sigma
        mid_vals.append(dt[m_mid]); mid_w.append(w[m_mid])
        far_vals.append(dt[m_far]); far_w.append(w[m_far])
    return mid_vals, mid_w, far_vals, far_w

mc_stack_mid, mc_weights_mid, mc_stack_far, mc_weights_far = split_mid_far(
    delta_t_mc_outside_ord, mc_weights_outside_ord, bunch_sigma
)

mask_data_mid = (np.abs(delta_t_data_outside) >= bunch_sigma) & (np.abs(delta_t_data_outside) < 2 * bunch_sigma)
mask_data_far = np.abs(delta_t_data_outside) >= 2 * bunch_sigma
mask_ext_mid  = (np.abs(delta_t_ext_outside)  >= bunch_sigma) & (np.abs(delta_t_ext_outside)  < 2 * bunch_sigma)
mask_ext_far  = np.abs(delta_t_ext_outside)  >= 2 * bunch_sigma

data_mid, data_far = delta_t_data_outside[mask_data_mid], delta_t_data_outside[mask_data_far]
ext_mid,  ext_far  = delta_t_ext_outside[mask_ext_mid],  delta_t_ext_outside[mask_ext_far]

# EXPECTED_SKYSHINE_OUTSIDE = EXPECTED_SKYSHINE * (298.4/182) factor).
EXPECTED_SKYSHINE_MID = EXPECTED_SKYSHINE                 # 1-2 sigma has the same exposure
EXPECTED_SKYSHINE_FAR = EXPECTED_SKYSHINE / (182/123.4)   # 2 sigma has the same as the skyshine sideband
print(EXPECTED_SKYSHINE_FAR, 'skyshine events in the >2-sigma window\n')
# -----------------------------------------------------------------------------

def plot_with_skyshine_3panel(
        bins, mc_data, on_beam_data, off_beam_data, skyshine_data,
        mc_data_mid, mc_weights_mid, on_beam_mid, off_beam_mid, skyshine_mid_data,
        mc_data_far, mc_weights_far, on_beam_far, off_beam_far, skyshine_far_data,
        pot_mc, pot_data, trig_ext, trig_data,
        expected_skyshine_yield, expected_skyshine_mid_yield, expected_skyshine_far_yield,
        axis_label, y_axis_label, x_limit, y_limit, y_limit_outside,
        xaxis_minor, yaxis_minor, yaxis_minor_outside,
        plot_name, legend_columns=3, mc_weights_by_cat=None, combined_errors=False):

    # ---- shared scaling factors ----
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    f_sky     = expected_skyshine_yield     / len(skyshine_data)     if len(skyshine_data)     > 0 else 0.0
    f_sky_mid = expected_skyshine_mid_yield / len(skyshine_mid_data) if len(skyshine_mid_data) > 0 else 0.0
    f_sky_far = expected_skyshine_far_yield / len(skyshine_far_data) if len(skyshine_far_data) > 0 else 0.0

    # ---- INSIDE (main panel) ----
    mc_weights = []
    for i, arr in enumerate(mc_data):
        genie_w = np.array(mc_weights_by_cat[i]) if mc_weights_by_cat is not None else np.ones(len(arr))
        mc_weights.append(genie_w * f_pot)

    prediction_data = [mc_data[0], skyshine_data] + mc_data[1:] + [off_beam_data]
    weights = ([mc_weights[0], np.ones(len(skyshine_data)) * f_sky]
               + mc_weights[1:] + [np.ones(len(off_beam_data)) * f_ext])

    # ---- MID (1-2 sigma) ----
    mid_mc_w  = [np.array(mc_weights_mid[i]) * f_pot for i in range(len(mc_data_mid))]
    mid_pred_data    = [mc_data_mid[0], skyshine_mid_data] + mc_data_mid[1:] + [off_beam_mid]
    mid_pred_weights = ([mid_mc_w[0], np.ones(len(skyshine_mid_data)) * f_sky_mid]
                         + mid_mc_w[1:] + [np.ones(len(off_beam_mid)) * f_ext])

    # ---- FAR (>2 sigma) ----
    far_mc_w  = [np.array(mc_weights_far[i]) * f_pot for i in range(len(mc_data_far))]
    far_pred_data    = [mc_data_far[0], skyshine_far_data] + mc_data_far[1:] + [off_beam_far]
    far_pred_weights = ([far_mc_w[0], np.ones(len(skyshine_far_data)) * f_sky_far]
                         + far_mc_w[1:] + [np.ones(len(off_beam_far)) * f_ext])

    # ---- aesthetics (matches plot_with_skyshine) ----
    labels_base = ['External (MC)', 'Skyshine', 'Out-of-FV', 'CC',
                    'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors      = ['papayawhip', 'darkmagenta', 'lime', 'darkred',
                    'cyan', 'darkorange', '#0066ff', 'gray']

    weights[0]          *= ALPHA_SCALING
    mid_pred_weights[0] *= ALPHA_SCALING
    far_pred_weights[0] *= ALPHA_SCALING

    # ---- main panel histogram math ----
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths  = np.diff(bins)

    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)

    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0]
                        for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0]
                        for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))
    plot_err = np.sqrt(data_err**2 + pred_err**2) if combined_errors else data_err

    # ---- helper: aggregate stats for a single-bin "outside" panel ----
    def single_bin_stats(on_beam, pred_weights):
        data_count = len(on_beam)
        data_err_  = np.sqrt(data_count)
        yields     = [np.sum(w) for w in pred_weights]
        pred_total = sum(yields)
        pred_err_  = np.sqrt(sum(np.sum(w**2) for w in pred_weights))
        plot_err_  = np.sqrt(data_err_**2 + pred_err_**2) if combined_errors else data_err_
        return data_count, yields, pred_total, pred_err_, plot_err_

    mid_data_count, mid_yields, mid_pred_total, mid_pred_err, mid_plot_err = \
        single_bin_stats(on_beam_mid, mid_pred_weights)
    far_data_count, far_yields, far_pred_total, far_pred_err, far_plot_err = \
        single_bin_stats(on_beam_far, far_pred_weights)

    # legend labels with %, based on main-panel yields
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]

    # ------------------------------------------------------------------ #
    # Layout: wide main | mid (1-2 sigma) | far (>2 sigma), residual row each
    # ------------------------------------------------------------------ #
    fig = plt.figure(figsize=(10, 6))
    gs  = fig.add_gridspec(2, 5, height_ratios=[4, 1],
                            width_ratios=[10, 0.4, 1.7, 0.12, 1.7],
                            hspace=0.1, wspace=0.08)

    ax      = fig.add_subplot(gs[0, 0])
    ax2     = fig.add_subplot(gs[1, 0])
    ax_mid  = fig.add_subplot(gs[0, 2])
    ax_mid2 = fig.add_subplot(gs[1, 2])
    ax_far  = fig.add_subplot(gs[0, 4])
    ax_far2 = fig.add_subplot(gs[1, 4])

    # ---- main panel: alpha fill + hatch MC-stat band + bold total outline ----
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights,
            label=labels, color=colors, linewidth=0, alpha=0.65)

    x_step    = np.repeat(bins, 2)[1:-1]
    y_band_hi = np.repeat(pred_counts + pred_err, 2)
    y_band_lo = np.repeat(np.maximum(pred_counts - pred_err, 0), 2)
    ax.fill_between(x_step, y_band_lo, y_band_hi,
                     facecolor='none', edgecolor='dimgray',
                     linewidth=0, hatch='//', zorder=3, label='MC stats')
    ax.stairs(pred_counts, bins, color='black', linewidth=1.8, zorder=4)

    ax.errorbar(bin_centers, data_counts, yerr=plot_err, xerr=bin_widths/2,
                fmt='ko', label='On-Beam Data (stats only)', markersize=4, capsize=0)

    # ---- main residual ----
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    ratio  = data_counts[mask] / pred_counts[mask]
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask],
                 xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)

    # ---- helper: draw one "outside" stacked-bar panel + its residual ----
    def draw_outside_panel(ax_top, ax_bot, yields_, pred_total_, pred_err_,
                            data_count_, plot_err_, tick_label):
        bar_x, bar_w = 0.5, 0.7
        bottom = 0.0
        for yv, col in zip(yields_, colors):
            ax_top.bar(bar_x, yv, bar_w, bottom=bottom,
                       color=col, edgecolor='none', linewidth=0, alpha=0.65)
            bottom += yv

        out_total = pred_total_ if pred_total_ > 0 else 1
        bottom = 0.0
        for yv, col in zip(yields_, colors):
            if yv / out_total > 0.05:
                ax_top.text(bar_x, bottom + yv/2, f'{100*yv/out_total:.1f}%',
                            ha='center', va='center', fontsize=8, color='black')
            bottom += yv

        # single-bin analogue of the main panel's hatch band + stairs outline
        lo, hi = max(pred_total_ - pred_err_, 0), pred_total_ + pred_err_
        ax_top.fill_between([bar_x - bar_w/2, bar_x + bar_w/2], lo, hi,
                             facecolor='none', edgecolor='dimgray',
                             linewidth=0, hatch='//', zorder=3)
        ax_top.plot([bar_x - bar_w/2, bar_x + bar_w/2], [pred_total_, pred_total_],
                    color='black', linewidth=1.8, zorder=4)

        ax_top.errorbar(bar_x, data_count_, yerr=plot_err_, fmt='ko', markersize=4, capsize=0)

        if pred_total_ > 0:
            res_     = (data_count_ - pred_total_) / pred_total_
            ratio_   = data_count_ / pred_total_
            d_term_  = (np.sqrt(data_count_)/data_count_)**2 if data_count_ > 0 else 0
            res_err_ = ratio_ * np.sqrt(d_term_ + (pred_err_/pred_total_)**2)
            ax_bot.errorbar(bar_x, res_, yerr=res_err_, fmt='ko', markersize=4)

        ax_bot.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax_bot.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
        ax_bot.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)

        ax_top.set_xlim([0, 1]); ax_top.tick_params(axis='x', labelbottom=False); ax_top.set_xticks([])
        ax_bot.set_xlim([0, 1]); ax_bot.set_ylim([-1.2, 1.2])
        ax_bot.set_xticks([0.5]); ax_bot.set_xticklabels([tick_label], fontsize=14)

    draw_outside_panel(ax_mid, ax_mid2, mid_yields, mid_pred_total, mid_pred_err,
                        mid_data_count, mid_plot_err, r'$1$-$2\sigma_{\nu\mathrm{NCQE}}$')
    draw_outside_panel(ax_far, ax_far2, far_yields, far_pred_total, far_pred_err,
                        far_data_count, far_plot_err, r'$>2\sigma_{\nu\mathrm{NCQE}}$')

    ax_mid.set_ylim(y_limit_outside)
    ax_mid.yaxis.set_minor_locator(MultipleLocator(yaxis_minor_outside))
    ax_mid.tick_params(axis='y', labelleft=False)    # shares far panel's scale — no duplicate labels
    ax_mid2.tick_params(axis='y', labelleft=False)   # same for the residual row
    
    ax_far.set_ylim(y_limit_outside)
    ax_far.yaxis.set_minor_locator(MultipleLocator(yaxis_minor_outside))
    ax_far.yaxis.tick_right()                          # move ticks to the right spine
    ax_far.tick_params(axis='y', labelright=True, labelleft=False)
    
    ax_far2.yaxis.tick_right()                          # same for the residual row
    ax_far2.tick_params(axis='y', labelright=True, labelleft=False)


    # ---- formatting ----
    ax.set_ylabel(y_axis_label, fontsize=16)
    ax.set_xlim(x_limit); ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    ax.yaxis.set_minor_locator(MultipleLocator(yaxis_minor))

    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=16)
    ax2.set_xlabel(axis_label, fontsize=16)
    ax2.set_xlim(x_limit); ax2.set_ylim([-1.2, 1.2])
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))

    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.60, 1.02, f"{pot_data:.2f}" + r" $\times 10^{20}$ POT", transform=ax.transAxes)

    plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()
    
    
plot_with_skyshine_3panel(
    #bins=np.array([-bunch_sigma, -2.5, -1.5, -0.5, 0.5, 1.5, 2.5, bunch_sigma]),
    bins=np.array([-bunch_sigma, -2.5, -1, 1, 2.5, bunch_sigma]),
    mc_data=mc_stack,
    on_beam_data=delta_t_data,
    off_beam_data=delta_t_ext,
    skyshine_data=skyshine_delta_t_bunch,

    mc_data_mid=mc_stack_mid, mc_weights_mid=mc_weights_mid,
    on_beam_mid=data_mid, off_beam_mid=ext_mid, skyshine_mid_data=skyshine_delta_t_bunch,

    mc_data_far=mc_stack_far, mc_weights_far=mc_weights_far,
    on_beam_far=data_far, off_beam_far=ext_far, skyshine_far_data=skyshine_delta_t_bunch,

    pot_mc=POT_MC, pot_data=POT_DATA, trig_ext=TRIG_EXT, trig_data=TRIG_DATA,
    expected_skyshine_yield=EXPECTED_SKYSHINE,
    expected_skyshine_mid_yield=EXPECTED_SKYSHINE_MID,
    expected_skyshine_far_yield=EXPECTED_SKYSHINE_FAR,
    axis_label=r'$\Delta t_{\mathrm{nearest\ bunch}}$ [ns]',
    y_axis_label='Events / bin',
    x_limit=[-bunch_sigma, bunch_sigma],
    y_limit=[0, 325],
    y_limit_outside=[0, 650],
    xaxis_minor=0.5,
    yaxis_minor=10,
    yaxis_minor_outside=50,
    plot_name=plots_dir + 'full_data_delta_t_3panel.png',
    legend_columns=3,
    mc_weights_by_cat=mc_weights_stack,
)

print('done')

In [ ]:
# EVENT RATE BREAKDOWN

# Breakdown of the event category counts (w/ weighting applied)
# (pass this along to the XS extraction script)

pot_scale = POT_DATA / POT_MC
trig_scale = TRIG_DATA / TRIG_EXT

cat_labels = ['NCQE (signal)', 'NCother', 'CC', 'nubarNCQE', 'OOFV', 'External', 'Off-beam', 'Skyshine']

# From earlier MC truth cell
MC_TRUE_ABS_SIGNAL = 19442        # un-scaled to data POT

print('='*30)
print('MC RAW COUNTS')
print('='*30)
print('MC POT:', POT_MC)
print('Total NCQE true signal:', MC_TRUE_ABS_SIGNAL)
print('-'*20)

# MC categories (with weights)
counter = 0
for i in range(len(MC_weights_by_cat)):
    if i == 5:
        print(cat_labels[i], round(ALPHA_SCALING*sum(MC_weights_by_cat[i]), 2))
        counter += ALPHA_SCALING*sum(MC_weights_by_cat[i])
    else:
        print(cat_labels[i], round(sum(MC_weights_by_cat[i]), 2))
        counter += sum(MC_weights_by_cat[i])

print('Off-beam:', len(ext_reco_E)*trig_scale); counter += len(ext_reco_E)*trig_scale
print('Skyshine:', EXPECTED_SKYSHINE); counter += EXPECTED_SKYSHINE
print('-'*20)
print('Total Predicted:', counter)

# # # # # # # # # # # # # # # # # # # # # #

print('\n')
print('='*30)
print('POT SCALED')
print('='*30)
print('MC POT:', POT_MC)
print('Total NCQE true signal:', MC_TRUE_ABS_SIGNAL*(POT_DATA/POT_MC))
print('-'*20)

# MC categories (with weights)
counter = 0
for i in range(len(MC_weights_by_cat)):
    if i == 5:
        print(cat_labels[i], round(ALPHA_SCALING*(POT_DATA/POT_MC)*sum(MC_weights_by_cat[i]), 2))
        counter += ALPHA_SCALING*(POT_DATA/POT_MC)*sum(MC_weights_by_cat[i])
    else:
        print(cat_labels[i], round((POT_DATA/POT_MC)*sum(MC_weights_by_cat[i]), 2))
        counter += (POT_DATA/POT_MC)*sum(MC_weights_by_cat[i])

print('Off-beam:', len(ext_reco_E)*trig_scale); counter += len(ext_reco_E)*trig_scale
print('Skyshine:', EXPECTED_SKYSHINE); counter += EXPECTED_SKYSHINE
print('-'*20)
print('Total Predicted:', counter)
print('Total Observed:', len(reco_E))

In [ ]:
# Lastly, export the file for easier comparisons between Base / Empirical GENIE

# Lastly, export the file for easier comparisons between Base / Empirical GENIE

root_path = "Empirical_GENIE.root"  # "Base_GENIE.root"

category_names = {
    0: "nuNCQE",
    1: "NCother",
    2: "CC",
    3: "nubarNCQE",
    4: "outOfFV",
    5: "External",
}

def event_branches(
    reco_E, hits, cv_x, cv_y, cv_z,
    bunch_time, vertex_x, vertex_y, vertex_z,
    radius, delta_t, weights=None
):
    branches = {
        "reco_E":   np.asarray(reco_E, dtype="f8"),
        "hits":     np.asarray(hits, dtype="i8"),
        "CV_x":     np.asarray(cv_x, dtype="f8"),
        "CV_y":     np.asarray(cv_y, dtype="f8"),
        "CV_z":     np.asarray(cv_z, dtype="f8"),
        "bunch_time": np.asarray(bunch_time, dtype="f8"),
        "vertex_x": np.asarray(vertex_x, dtype="f8"),
        "vertex_y": np.asarray(vertex_y, dtype="f8"),
        "vertex_z": np.asarray(vertex_z, dtype="f8"),
        "radius":   np.asarray(radius, dtype="f8"),
        "delta_t":  np.asarray(delta_t, dtype="f8"),
    }

    if weights is not None:
        branches["genie_weight"] = np.asarray(weights, dtype="f8")

    return branches


def sigma_region_branches(bunch_time, delta_t, weights=None):
    """Branch set for events outside the 1-sigma cut, where only bunch_time /
    delta_t (and MC weight) were ever recorded upstream."""
    branches = {
        "bunch_time": np.asarray(bunch_time, dtype="f8"),
        "delta_t":    np.asarray(delta_t, dtype="f8"),
    }
    if weights is not None:
        branches["genie_weight"] = np.asarray(weights, dtype="f8")
    return branches


def sigma_masks(delta_t, sigma):
    """boolean masks for the 1-2 sigma ('mid') and >2 sigma ('far') sub-regions"""
    dt = np.abs(np.asarray(delta_t))
    return (dt >= sigma) & (dt < 2 * sigma), dt >= 2 * sigma


with uproot.recreate(root_path) as root_file:

    # MC, stored in the original category numbering
    for i, name in category_names.items():
        root_file[f"mc_{name}"] = event_branches(
            MC_reco_E[i],
            MC_hits[i],
            MC_CV_x[i], MC_CV_y[i], MC_CV_z[i],
            MC_bunch_times[i],
            MC_vertex_x[i], MC_vertex_y[i], MC_vertex_z[i],
            radius_mc[i],
            delta_t_mc[i],
            weights=MC_weights_by_cat[i],
        )

    # On-beam data
    root_file["onbeam"] = event_branches(
        reco_E, hits,
        CV_x, CV_y, CV_z,
        bunch_times,
        vertex_x, vertex_y, vertex_z,
        radius_data,
        delta_t_data,
    )

    # Off-beam data
    root_file["offbeam"] = event_branches(
        ext_reco_E, ext_hits,
        ext_CV_x, ext_CV_y, ext_CV_z,
        ext_bunch_times,
        ext_vertex_x, ext_vertex_y, ext_vertex_z,
        radius_ext,
        delta_t_ext,
    )

    # Skyshine
    # NOTE: skyshine_delta_t_bunch is a uniform-random sample generated only
    # within +/- bunch_sigma (see its construction cell) -- there is no
    # simulated skyshine sample for the outside/1-2sigma/>2sigma regions, so
    # no skyshine_outside/mid/far trees are written. EXPECTED_SKYSHINE_MID and
    # EXPECTED_SKYSHINE_FAR are still placeholder scale factors too.
    root_file["skyshine"] = {
        "reco_E": np.asarray(skyshine_reco_E, dtype="f8"),
        "hits": np.asarray(skyshine_hits, dtype="i8"),
        "CV_x": np.asarray(skyshine_CV_x, dtype="f8"),
        "CV_y": np.asarray(skyshine_CV_y, dtype="f8"),
        "CV_z": np.asarray(skyshine_CV_z, dtype="f8"),
        "bunch_time": np.asarray(skyshine_bunch_times, dtype="f8"),
        "vertex_x": np.asarray(skyshine_vertex_x, dtype="f8"),
        "vertex_y": np.asarray(skyshine_vertex_y, dtype="f8"),
        "vertex_z": np.asarray(skyshine_vertex_z, dtype="f8"),
        "radius": np.asarray(radius_skyshine, dtype="f8"),
        "delta_t": np.asarray(skyshine_delta_t_bunch, dtype="f8"),
    }

    # MC outside-1-sigma samples (full outside bucket, now with delta_t added)
    # + explicit 1-2 sigma ("mid") and >2 sigma ("far") sub-samples
    for i, name in category_names.items():
        root_file[f"mc_outside_{name}"] = sigma_region_branches(
            MC_outside_bunch_by_cat[i],
            delta_t_mc_outside[i],
            weights=MC_outside_weights_by_cat[i],
        )

        mask_mid, mask_far = sigma_masks(delta_t_mc_outside[i], bunch_sigma)
        bt = np.asarray(MC_outside_bunch_by_cat[i], dtype="f8")
        dt = np.asarray(delta_t_mc_outside[i], dtype="f8")
        w  = np.asarray(MC_outside_weights_by_cat[i], dtype="f8")

        root_file[f"mc_mid_{name}"] = sigma_region_branches(bt[mask_mid], dt[mask_mid], weights=w[mask_mid])
        root_file[f"mc_far_{name}"] = sigma_region_branches(bt[mask_far], dt[mask_far], weights=w[mask_far])

    # On-beam / off-beam: full outside-1-sigma bucket + 1-2 sigma / >2 sigma splits
    mask_data_mid, mask_data_far = sigma_masks(delta_t_data_outside, bunch_sigma)
    mask_ext_mid,  mask_ext_far  = sigma_masks(delta_t_ext_outside,  bunch_sigma)

    outside_bunch_times_arr     = np.asarray(outside_bunch_times, dtype="f8")
    ext_outside_bunch_times_arr = np.asarray(ext_outside_bunch_times, dtype="f8")

    root_file["onbeam_outside"] = sigma_region_branches(outside_bunch_times_arr, delta_t_data_outside)
    root_file["onbeam_mid"] = sigma_region_branches(
        outside_bunch_times_arr[mask_data_mid], delta_t_data_outside[mask_data_mid]
    )
    root_file["onbeam_far"] = sigma_region_branches(
        outside_bunch_times_arr[mask_data_far], delta_t_data_outside[mask_data_far]
    )

    root_file["offbeam_outside"] = sigma_region_branches(ext_outside_bunch_times_arr, delta_t_ext_outside)
    root_file["offbeam_mid"] = sigma_region_branches(
        ext_outside_bunch_times_arr[mask_ext_mid], delta_t_ext_outside[mask_ext_mid]
    )
    root_file["offbeam_far"] = sigma_region_branches(
        ext_outside_bunch_times_arr[mask_ext_far], delta_t_ext_outside[mask_ext_far]
    )

print(f"Wrote {root_path}")